# Test Utilities

> Helper functions that build an in-memory / temp-directory fixture replicating the remote IEMOCAP data sources (SQLite DB, Parquet audio chunks, JSON manifest) without making any network requests.

In [ ]:
#| default_exp test_utils

In [ ]:
#| export

import io
import json
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

## `create_test_fixtures`

Returns a `(tmpdir, tests_path)` pair where:

- **`tmpdir`** — a `tempfile.TemporaryDirectory` instance. Keep a reference to it for the lifetime of your test; the directory (and all its contents) are cleaned up automatically when it is garbage-collected or when you call `tmpdir.cleanup()`.
- **`tests_path`** — a `pathlib.Path` pointing to the temporary directory root, pre-populated with:
  - `dataset_audio_chunk_groups.json` — one datasetchunk-group record
  - `group_id_1__last_export_1778968883.parquet` — three audio_chunk rows backed by a 10-second 440 Hz sine wave
  - `iemocap_dataset_table.db` — a minimal SQLite DB with one audio row


In [ ]:
#| export

def create_test_fixtures() -> tuple[tempfile.TemporaryDirectory, Path]:
    """Create a temporary directory pre-populated with stub IEMOCAP data files.

    The fixtures mirror the real remote data sources so that DatasetsFactory
    can be initialised without any network access.

    Returns
    -------
    tmpdir : tempfile.TemporaryDirectory
        Holds a reference to the temp directory.  Call ``tmpdir.cleanup()``
        (or let it go out of scope) to remove all files when done.
    tests_path : Path
        Absolute path to the temp directory root.
    """
    # ── Fake audio: 10 s of 440 Hz sine wave at 16 kHz ──────────────────────
    sr = 16_000
    t = np.linspace(0, 10, sr * 10, endpoint=False)
    fake_audio = np.sin(2 * np.pi * 440 * t).astype(np.float32)

    buf = io.BytesIO()
    sf.write(buf, fake_audio, sr, format='WAV')
    audio_bytes = buf.getvalue()

    tmpdir = tempfile.TemporaryDirectory()
    tests_path = Path(tmpdir.name)

    # ── 1. JSON manifest ─────────────────────────────────────────────────────
    manifest_data = [
        {
            "id": 1,
            "chunk_threshold_seconds": 4.8,
            "previous_overlap_seconds": 0.2,
            "sample_rate": 16000,
            "last_export_filename": (
                "http://example.com/dataset_audio_chunks/"
                "group_id_1__last_export_1778968883.parquet"
            ),
            "last_export_at": "2026-05-16 22:01:23.000000",
        }
    ]
    from nbdev_upc_aidl_iemocap_datasets.core import DatasetsFactory
    with open(tests_path / DatasetsFactory.JSON_FILE_NAME, 'w') as f:
        json.dump(manifest_data, f)

    # ── 2. Parquet audio chunk file ────────────────────────────────────────────────
    _common = dict(
        source_dataset_table_name="iemocap_dataset",
        source_dataset_row_index=11,
        source_dataset_audio_filename="Ses01F_impro01_F011.wav",
       chunk_compressed_audio_url=None,
        source_duration_seconds=9.81,
        source_total_samples=156960,
        total_chunk_parts=3,
        assigned_reviewer_id=None,
        last_reviewer_id=None,
        reviewed_at=None,
        partition_type=0,
        reviewed_frustrated=0.635,
        reviewed_angry=0.320,
        reviewed_sad=0.006,
        reviewed_disgust=0.006,
        reviewed_excited=0.006,
        reviewed_fear=0.006,
        reviewed_neutral=0.006,
        reviewed_surprise=0.006,
        reviewed_happy=0.006,
        reviewed_major_emotion="frustrated",
        speaker_id="Ses01__F",
        track_group_id="Ses01F_impro01",
        track_group_speaker_turn=11
    )
    parquet_data = [
        {**_common, 'id': 13, 'dataset_audio_chunk_group_id': '1',
         'chunk_part_idx': 0, 'start_sample_offset': 0,
         'end_sample_offset': 76800, 'sample_duration': 4.8,
         'should_exclude': False, 'should_add_padding': False},
        {**_common, 'id': 14, 'dataset_audio_chunk_group_id': '1',
         'chunk_part_idx': 1, 'start_sample_offset': 73600,
         'end_sample_offset': 150400, 'sample_duration': 4.8,
         'should_exclude': False, 'should_add_padding': False},
        {**_common, 'id': 15, 'dataset_audio_chunk_group_id': '1',
         'chunk_part_idx': 2, 'start_sample_offset': 147200,
         'end_sample_offset': 156959, 'sample_duration': 0.609,
         'should_exclude': False, 'should_add_padding': True},
        {**_common, 'id': 1, 'dataset_audio_chunk_group_id': '1',
         'chunk_part_idx': 0, 'start_sample_offset': 0,
         'end_sample_offset': 1, 'sample_duration': 0.6,
         'should_exclude': True, 'should_add_padding': True},
    ]
    pd.DataFrame(parquet_data).to_parquet(
        tests_path / 'group_id_1__last_export_1778968883.parquet'
    )

    # ── 3. SQLite database ───────────────────────────────────────────────────
    db_path = tests_path / DatasetsFactory.IEMOCAP_DB_FILE_NAME
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE iemocap_dataset (
            row_index           INTEGER PRIMARY KEY,
            compressed_audio_url VARCHAR NULL,
            file                VARCHAR,
            audio_bytes         BLOB,
            frustrated FLOAT, angry FLOAT, sad FLOAT, disgust FLOAT,
            excited    FLOAT, fear  FLOAT, neutral FLOAT, surprise FLOAT,
            happy FLOAT,
            "EmoAct" FLOAT, "EmoVal" FLOAT, "EmoDom" FLOAT,
            gender         VARCHAR,
            transcription  VARCHAR,
            major_emotion  VARCHAR,
            speaking_rate  FLOAT,
            pitch_mean     FLOAT,
            pitch_std      FLOAT,
            rms            FLOAT,
            relative_db    FLOAT,
            speaker_id VARCHAR,
            track_group_id VARCHAR,
            track_group_speaker_turn INTEGER
        )
    """)
    conn.execute("""
        INSERT INTO iemocap_dataset VALUES
            (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        11, None, 'Ses01F_impro01_F011.wav', audio_bytes,
        0.635, 0.320, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006,
        3.666, 2.0, 3.666,
        'Female', 'Yes but my wallet was stolen...', 'frustrated',
        12.02, 266.14, 100.39, 0.064, -17.26, 'Ses01_F', 'Ses01F_impro01', 11
    ))
    conn.commit()
    conn.close()

    return tmpdir, tests_path
